In [ ]:
import pandas as pd
df = pd.read_csv("../data/data.csv")
df.head()

In [ ]:
df.shape
df.columns
df.dtypes

In [ ]:
df.describe()

In [ ]:
cat = df.select_dtypes(include="object").columns
for col in cat:
    print(df[col].value_counts())

In [ ]:
df.isna().sum()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.histplot(data=df, x="Exam_Score", bins=20, kde=True)
plt.title("scores d'examen")
plt.xlabel("Exam Score")
plt.ylabel("Nombre d'etudiants")
plt.show()

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

sns.histplot(data=df, x="Hours_Studied", bins=20, kde=True)
plt.title("houres studied")
plt.xlabel("Houres")
plt.ylabel("Nombre d'etudiants")
plt.show()

In [ ]:
corr = df.select_dtypes(include="number").corr()
sns.heatmap(corr, annot=True)
plt.title("Matrice de correlation")
plt.show()

In [ ]:
df_encoded = pd.get_dummies(df, drop_first=True) 
correlation_matrix = df_encoded.corr()
plt.figure(figsize=(15, 12))
sns.heatmap(
    correlation_matrix, 
    annot=True,  
    fmt=".2f",
)
plt.title("Heatmap de la matrice de correlation Apres Encodage")
plt.show()
correlation_matrix

In [ ]:
df.drop_duplicates()


In [ ]:
df.isna().sum()
df["Teacher_Quality"]=df["Teacher_Quality"].fillna(df["Teacher_Quality"].mode()[0])
df["Parental_Education_Level"]=df["Parental_Education_Level"].fillna(df["Parental_Education_Level"].mode()[0])
df["Distance_from_Home"]=df["Distance_from_Home"].fillna(df["Distance_from_Home"].mode()[0])
df.isna().sum()
df


In [ ]:
# sns.boxenplot(data= df, x=df["Exam_Score"])
from scipy import stats

Q1 = df["Exam_Score"].quantile(0.25)
Q3 = df["Exam_Score"].quantile(0.75)
IQR = Q3 - Q1

inf = Q1 - 1.5 * IQR
sup = Q3 + 1.5 * IQR

outliers = df[(df["Exam_Score"]<inf) | (df["Exam_Score"]>sup)]
df.drop(outliers.index, inplace=True)

df

# df["Zscore"] = stats.zscore(df["Exam_Score"])
# outliers_zscore = df[df['Zscore'].abs() > 3]

# outliers_zscore
sns.boxenplot(x=df["Exam_Score"])

In [ ]:
nominal_var = ["Gender","School_Type","Extracurricular_Activities","Internet_Access", "Learning_Disabilities", "Peer_Influence"]
df = pd.get_dummies(df, columns=nominal_var, drop_first=True)
df.head()
df.shape





In [ ]:
elems = ["Parental_Involvement", "Access_to_Resources", "Motivation_Level", "Family_Income", "Teacher_Quality", "Distance_from_Home", "Parental_Education_Level"]

ordinal_maps = {}
for el in elems:
    ordinal_maps[el] = {
            "Low": 0,
            "Medium": 1,
            "High": 2
    }
ordinal_maps["Distance_from_Home"] = {
    "Near": 0,
    "Moderate": 1,
    "Far": 2
}
ordinal_maps["Parental_Education_Level"] = {
    "High School": 0,
    "College": 1,
    "Postgraduate": 2,
}
print(ordinal_maps)
for el in elems:
    df[el] = df[el].map(ordinal_maps[el])
df["Parental_Education_Level"]


In [ ]:
def encoder_Nominal(variables):
    df = pd.get_dummies(df, columns=variables, drop_first=True)
def encoder_Ordinal(variables):



In [ ]:
df["Distance_from_Home"].unique()
df

In [ ]:
from sklearn.model_selection import train_test_split
x = df.drop("Exam_Score", axis=1)
y = df["Exam_Score"]
x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=24
)
num_cols = x.select_dtypes(include="number").columns

from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# scaler.fit(x_train)
# x_train[num_cols] = scaler.fit_transform(x_train[num_cols])
# x_test[num_cols] = scaler.transform(x_test[num_cols])

# x_train
from sklearn.compose import ColumnTransformer


preprocessor = ColumnTransformer(
    transformers=[
        ("scaler", StandardScaler(), num_cols)
    ],
    remainder="passthrough"
)


from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline
# model = LinearRegression()
# model.fit(x_train, y_train)
# pred = model.predict(x_test)
# pred

from sklearn.ensemble import RandomForestRegressor
# model = RandomForestRegressor()
# model.fit(x_train, y_train)
# pred = model.predict(x_test)
# pred


from xgboost import XGBRegressor
# model = XGBRegressor()
# model.fit(x_train, y_train)
# pred = model.predict(x_test)
# pred

from sklearn.svm import SVR
# model = SVR()
# model.fit(x_train, y_train)
# pred = model.predict(x_test)
# pred

models = {
    "Linear Regression": LinearRegression(),
    "Random Forest": RandomForestRegressor(),
    "XGBoost": XGBRegressor(),
    "SVR": SVR()
}
pipes = {}

for name, model in models.items():
    pipes[name] = Pipeline([
        ("preprocessor", preprocessor),
        ("model", model)
    ])
for name, pip in pipes.items():
    pip.fit(x_train, y_train)
predicts = {}

for name, pip in pipes.items():
    predicts[name] = pip.predict(x_test)

print(predicts)

In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
import numpy as np
import pandas as pd
evalute = []
for name, pip in pipes.items():
    y_pred = pip.predict(x_test)
    rmse = np.sqrt(mean_squared_error(y_test, y_pred))
    mae = mean_absolute_error(y_test, y_pred)
    r2 = r2_score(y_test, y_pred)
    evalute.append({
        "model": name,
        "rmse":rmse,
        "mae": mae,
        "r2": r2
    })
evaluation_df = pd.DataFrame(evalute)
evaluation_df


In [ ]:
param_grids = {
    "Linear Regression": {
        "model__fit_intercept": [True, False],
        "model__positive": [True, False]
    },

    "Random Forest": {
        "model__n_estimators": [100, 200],
        "model__max_depth": [3,5,7,10],
        "model__min_samples_split": [2, 5]
    },

    "XGBoost": {
        "model__n_estimators": [100, 200],
        "model__learning_rate": [0.05, 0.1],
        "model__max_depth": [3, 5],
        "model__subsample": [0.8, 1.0]
    },

    "SVR": {
        "model__C": [0.1, 1, 10],
        "model__kernel": ["linear", "rbf"],
        "model__epsilon": [0.01, 0.1]
    }
}

In [ ]:
from sklearn.model_selection import GridSearchCV
import pandas as pd
best_models = {}
cv_results = []
for name, pip in pipes.items():
    grid = GridSearchCV(
        estimator=pip,
        param_grid=param_grids[name],
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    grid.fit(x_train, y_train)
    best_models[name] = grid.best_estimator_
    cv_results.append({
        "Model": name,
        "Best R2": grid.best_score_,
        "Best Parameters": grid.best_params_
    })
results_cv = pd.DataFrame(cv_results)
results_cv


In [ ]:
results_cv.loc[0 ,'Best Parameters']

In [ ]:
results = []
for name in pipes.keys():
    pred_before = pipes[name].predict(x_test)
    pred_after = best_models[name].predict(x_test)
    results.append({
        "Model": name,
        "RMSE Before": np.sqrt(mean_squared_error(y_test, pred_before)),
        "RMSE After": np.sqrt(mean_squared_error(y_test, pred_after)),
        "MAE Before": mean_absolute_error(y_test, pred_before),
        "MAE After": mean_absolute_error(y_test, pred_after),
        "R2 Before": r2_score(y_test, pred_before),
        "R2 After": r2_score(y_test, pred_after)
    })

comparison = pd.DataFrame(results)
comparison

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np

for name, model in best_models.items():
    y_pred = model.predict(x_test)
    sns.scatterplot(x=y_test, y=y_pred, alpha=0.5)
    plt.title(f"{name} - Actual vs Predicted")
    plt.xlabel("Actual Exam Score")
    plt.ylabel("Predicted Exam Score")
    plt.show()

In [ ]:
for name, model in best_models.items():
    y_pred = model.predict(x_test)
    residuals = y_test.to_numpy() - y_pred
    sns.histplot(residuals, bins=30, kde=True)
    plt.title(f"{name} - Residual Distribution")
    plt.xlabel("Residuals")
    plt.ylabel("Frequency")
    plt.show()

In [ ]:
import pandas as pd
import numpy as np
from sklearn.metrics import mean_squared_error, mean_absolute_error, r2_score
results = []
for name, model in best_models.items():
    y_pred = model.predict(x_test)
    residuals = y_test.to_numpy() - y_pred
    results.append({
        "Model": name,
        "RMSE": np.sqrt(mean_squared_error(y_test, y_pred)),
        "MAE": mean_absolute_error(y_test, y_pred),
        "R2": r2_score(y_test, y_pred),
        "ResiStd": np.std(residuals)
    })
results_df = pd.DataFrame(results)
results_df

In [ ]:
from sklearn.model_selection import cross_val_score
cv_results = []
for name, model in best_models.items():
    scores = cross_val_score(
        model,
        x_train,
        y_train,
        cv=3,
        scoring="r2",
        n_jobs=-1
    )
    cv_results.append({
        "Model": name,
        "CV R2 Mean": scores.mean(),
        "CV R2 STD": scores.std()
    })
cv_df = pd.DataFrame(cv_results)
cv_df

In [ ]:
selected_name = cv_df.loc[cv_df["CV R2 Mean"].idxmax(),"Model"]
final_model = best_models[selected_name]
print("Selected model:", selected_name)

In [ ]:
import joblib
joblib.dump(final_model, "../models/edupredict_pipeline.joblib")
joblib.dump(x_train.columns.tolist(),"../models/feature_columns.joblib")

In [2]:
import pandas as pd
df = pd.DataFrame({
    "experience": [1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16],

    "departement": [
        "IT", "RH", "IT", "Finance",
        "RH", "IT", "Finance", "IT",
        "RH", "Finance", "IT", "RH",
        "Finance", "IT", "Finance", "RH"
    ],

    "niveau": [
        "Junior", "Junior", "Intermediaire", "Junior",
        "Intermediaire", "Intermediaire", "Senior", "Intermediaire",
        "Senior", "Senior", "Expert", "Senior",
        "Expert", "Expert", "Expert", "Senior"
    ],

    "salaire_kdh": [
        6, 5, 8, 7,
        7, 10, 9, 12,
        9, 11, 14, 11,
        13, 16, 15, 13
    ]
})




In [ ]:
df
from sklearn.model_selection import train_test_split
X_

,experience,departement,niveau,salaire_kdh
0,1,IT,Junior,6
1,2,RH,Junior,5
2,3,IT,Intermediaire,8
3,4,Finance,Junior,7
4,5,RH,Intermediaire,7
5,6,IT,Intermediaire,10
6,7,Finance,Senior,9
7,8,IT,Intermediaire,12
8,9,RH,Senior,9
9,10,Finance,Senior,11


In [10]:
from sklearn.model_selection import train_test_split, GridSearchCV
x = df[["experience"]]
y=df["salaire_kdh"]

x_train, x_test, y_train, y_test = train_test_split(
    x,
    y,
    test_size=0.2,
    random_state=25
)

from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.pipeline import Pipeline

pip = Pipeline([
     ("scaler", StandardScaler()),
     ("model", LinearRegression())
])
grid_pa =    {
        "model__fit_intercept": [True, False],
        "model__positive": [True, False]
    }
grid = GridSearchCV(
    estimator=pip,
    param_grid=grid_pa,
    cv=2,
    scoring="r2",
    n_jobs=-1
)
grid.fit(x_train, y_train)




,"estimator estimator: estimator objectThis is assumed to implement the scikit-learn estimator interface.Either estimator needs to provide a ``score`` function,or ``scoring`` must be passed.",Pipeline(step...egression())])
,"param_grid param_grid: dict or list of dictionariesDictionary with parameters names (`str`) as keys and lists ofparameter settings to try as values, or a list of suchdictionaries, in which case the grids spanned by each dictionaryin the list are explored. This enables searching over any sequenceof parameter settings.","{'model__fit_intercept': [True, False], 'model__positive': [True, False]}"
,"scoring scoring: str, callable, list, tuple or dict, default=NoneStrategy to evaluate the performance of the cross-validated model onthe test set.If `scoring` represents a single score, one can use:- a single string (see :ref:`scoring_string_names`);- a callable (see :ref:`scoring_callable`) that returns a single value;- `None`, the `estimator`'s :ref:`default evaluation criterion <scoring_api_overview>` is used.If `scoring` represents multiple scores, one can use:- a list or tuple of unique strings;- a callable returning a dictionary where the keys are the metric names and the values are the metric scores;- a dictionary with metric names as keys and callables as values.See :ref:`multimetric_grid_search` for an example.",'r2'
,"n_jobs n_jobs: int, default=NoneNumber of jobs to run in parallel.``None`` means 1 unless in a :obj:`joblib.parallel_backend` context.``-1`` means using all processors. See :term:`Glossary <n_jobs>`for more details... versionchanged:: v0.20 `n_jobs` default changed from 1 to None",-1
,"cv cv: int, cross-validation generator or an iterable, default=NoneDetermines the cross-validation splitting strategy.Possible inputs for cv are:- None, to use the default 5-fold cross validation,- integer, to specify the number of folds in a `(Stratified)KFold`,- :term:`CV splitter`,- an iterable yielding (train, test) splits as arrays of indices.For integer/None inputs, if the estimator is a classifier and ``y`` iseither binary or multiclass, :class:`StratifiedKFold` is used. In allother cases, :class:`KFold` is used. These splitters are instantiatedwith `shuffle=False` so the splits will be the same across calls.Refer :ref:`User Guide <cross_validation>` for the variouscross-validation strategies that can be used here... versionchanged:: 0.22 ``cv`` default value if None changed from 3-fold to 5-fold.",2
,"refit refit: bool, str, or callable, default=TrueRefit an estimator using the best found parameters on the wholedataset.For multiple metric evaluation, this needs to be a `str` denoting thescorer that would be used to find the best parameters for refittingthe estimator at the end.Where there are considerations other than maximum score inchoosing a best estimator, ``refit`` can be set to a function whichreturns the selected ``best_index_`` given ``cv_results_``. In thatcase, the ``best_estimator_`` and ``best_params_`` will be setaccording to the returned ``best_index_`` while the ``best_score_``attribute will not be available.The refitted estimator is made available at the ``best_estimator_``attribute and permits using ``predict`` directly on this``GridSearchCV`` instance.Also for multiple metric evaluation, the attributes ``best_index_``,``best_score_`` and ``best_params_`` will only be available if``refit`` is set and all of them will be determined w.r.t this specificscorer.See ``scoring`` parameter to know more about multiple metricevaluation.See :ref:`sphx_glr_auto_examples_model_selection_plot_grid_search_digits.py`to see how to design a custom selection strategy using a callablevia `refit`.See :ref:`this example<sphx_glr_auto_examples_model_selection_plot_grid_search_refit_callable.py>`for an example of how to use ``refit=callable`` to balance modelcomplexity and cross-validated score... versionchanged:: 0.20 Support for callable added.",True
,"verbose verbose: int, default=0Controls the verbosity of